# Opening Effectiveness Analysis

This notebook analyzes chess opening performance metrics using a dataset of 20,058 games from Lichess.org.

**Objectives**
- Analyze win rates by opening
- Compare opening popularity vs effectiveness
- Identify top performing openings for White and Black

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

## 1. Data Preparation
Loading the chess games dataset from Lichess.org containing over 20,000 games.

In [ ]:
df = pd.read_csv('../data/games.csv')

df.info()
df.head()

## 2. Feature Selection
Selecting relevant columns for opening effectiveness analysis and creating a focused dataset.

In [ ]:
# Keep only the relevant columns
columns_to_keep = [
    'opening_name',
    'opening_eco',
    'winner',
    'white_rating',
    'black_rating',
    'turns',
    'victory_status'
]

opening_df = df[columns_to_keep].copy()

# Calculate rating difference
opening_df['rating_diff'] = opening_df['white_rating'] - opening_df['black_rating']
opening_df['abs_rating_diff'] = abs(opening_df['rating_diff'])

# Extract ECO family (first letter of ECO code)
opening_df['eco_family'] = opening_df['opening_eco'].str[0]

opening_df.info()
opening_df.head()

## 3. Opening Popularity Analysis
Identifying the most and least popular openings in the dataset.

In [ ]:
# Calculate opening popularity
opening_popularity = opening_df['opening_name'].value_counts()
opening_popularity_pct = opening_df['opening_name'].value_counts(normalize=True) * 100

print("="*70)
print("OPENING POPULARITY ANALYSIS")
print("="*70)
print(f"\nTotal unique openings: {len(opening_popularity)}")
print(f"Total games: {len(opening_df):,}")

print(f"\n{'='*70}")
print("TOP 20 MOST POPULAR OPENINGS")
print("="*70)
top_20 = pd.DataFrame({
    'Games': opening_popularity.head(20),
    'Percentage': opening_popularity_pct.head(20)
})
print(top_20)

# Visualize top 20 most popular openings
fig, ax = plt.subplots(figsize=(12, 8))

top_20_data = opening_popularity.head(20)
colors = plt.cm.viridis(np.linspace(0.3, 0.9, 20))

bars = ax.barh(range(len(top_20_data)), top_20_data.values, color=colors, edgecolor='black', linewidth=1)
ax.set_yticks(range(len(top_20_data)))
ax.set_yticklabels(top_20_data.index, fontsize=9)
ax.invert_yaxis()

# Add value labels
for i, (idx, value) in enumerate(top_20_data.items()):
    pct = opening_popularity_pct[idx]
    ax.text(value, i, f' {value} ({pct:.1f}%)', va='center', fontsize=9)

ax.set_xlabel('Number of Games', fontsize=12, fontweight='bold')
ax.set_ylabel('Opening Name', fontsize=12, fontweight='bold')
ax.set_title('Top 20 Most Popular Chess Openings', fontsize=14, fontweight='bold', pad=15)
sns.despine()

plt.tight_layout()
plt.savefig('../images/03_opening_effectiveness/opening_popularity.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# Aggregate statistics by opening
opening_outcomes = opening_df.groupby(['opening_name', 'winner']).size().unstack(fill_value=0)

opening_stats = pd.DataFrame({
    'opening_name': opening_outcomes.index,
    'eco_code': opening_df.groupby('opening_name')['opening_eco'].first(),
    'total_games': opening_outcomes.sum(axis=1)
})

# Calculate win rate percentages
for col in ['white', 'black', 'draw']:
    if col in opening_outcomes.columns:
        opening_stats[f'{col}_win_rate'] = (opening_outcomes[col] / opening_stats['total_games'] * 100)
    else:
        opening_stats[f'{col}_win_rate'] = 0.0

opening_stats = opening_stats.rename(columns={'draw_win_rate': 'draw_rate'})
opening_stats = opening_stats.reset_index(drop=True)

print("="*70)
print("OPENING STATISTICS SUMMARY")
print("="*70)
print(f"Total unique openings: {len(opening_stats)}")
print(f"\nSample (top 5 by total games):")
print(opening_stats.nlargest(5, 'total_games')[['opening_name', 'eco_code', 'total_games', 'white_win_rate', 'black_win_rate', 'draw_rate']].to_string(index=False))

## 4. Top Performing Openings
Identifying the best openings for White, Black, and most drawish openings.

In [ ]:
# Filter openings with at least 50 games for more reliable statistics
reliable_openings = opening_stats[opening_stats['total_games'] >= 50].copy()

# Top openings for White
top_white = reliable_openings.nlargest(10, 'white_win_rate')[['opening_name', 'eco_code', 'total_games', 'white_win_rate', 'black_win_rate', 'draw_rate']]

# Top openings for Black
top_black = reliable_openings.nlargest(10, 'black_win_rate')[['opening_name', 'eco_code', 'total_games', 'white_win_rate', 'black_win_rate', 'draw_rate']]

# Most drawish openings
most_drawish = reliable_openings.nlargest(10, 'draw_rate')[['opening_name', 'eco_code', 'total_games', 'white_win_rate', 'black_win_rate', 'draw_rate']]

print("="*70)
print("TOP 10 OPENINGS FOR WHITE (≥50 games)")
print("="*70)
print(top_white.to_string(index=False))

print(f"\n{'='*70}")
print("TOP 10 OPENINGS FOR BLACK (≥50 games)")
print("="*70)
print(top_black.to_string(index=False))

print(f"\n{'='*70}")
print("TOP 10 MOST DRAWISH OPENINGS (≥50 games)")
print("="*70)
print(most_drawish.to_string(index=False))

# Visualize top performing openings
fig, axes = plt.subplots(1, 3, figsize=(18, 8))

# Panel 1: Top openings for White
ax1 = axes[0]
top_white_plot = top_white.head(10).sort_values('white_win_rate')
bars1 = ax1.barh(range(len(top_white_plot)), top_white_plot['white_win_rate'], 
                 color='#E8E8E8', edgecolor='black', linewidth=1.5)
ax1.set_yticks(range(len(top_white_plot)))
ax1.set_yticklabels([name[:30] + '...' if len(name) > 30 else name for name in top_white_plot['opening_name']], fontsize=8)
ax1.set_xlabel('White Win Rate (%)', fontsize=11, fontweight='bold')
ax1.set_title('Top 10 Openings for White', fontsize=12, fontweight='bold')
for i, (idx, row) in enumerate(top_white_plot.iterrows()):
    ax1.text(row['white_win_rate'], i, f" {row['white_win_rate']:.1f}%", va='center', fontsize=9)
sns.despine(ax=ax1)

# Panel 2: Top openings for Black
ax2 = axes[1]
top_black_plot = top_black.head(10).sort_values('black_win_rate')
bars2 = ax2.barh(range(len(top_black_plot)), top_black_plot['black_win_rate'], 
                 color='#404040', edgecolor='black', linewidth=1.5)
ax2.set_yticks(range(len(top_black_plot)))
ax2.set_yticklabels([name[:30] + '...' if len(name) > 30 else name for name in top_black_plot['opening_name']], fontsize=8)
ax2.set_xlabel('Black Win Rate (%)', fontsize=11, fontweight='bold')
ax2.set_title('Top 10 Openings for Black', fontsize=12, fontweight='bold')
for i, (idx, row) in enumerate(top_black_plot.iterrows()):
    ax2.text(row['black_win_rate'], i, f" {row['black_win_rate']:.1f}%", va='center', fontsize=9)
sns.despine(ax=ax2)

# Panel 3: Most drawish openings
ax3 = axes[2]
most_drawish_plot = most_drawish.head(10).sort_values('draw_rate')
bars3 = ax3.barh(range(len(most_drawish_plot)), most_drawish_plot['draw_rate'], 
                 color='#808080', edgecolor='black', linewidth=1.5)
ax3.set_yticks(range(len(most_drawish_plot)))
ax3.set_yticklabels([name[:30] + '...' if len(name) > 30 else name for name in most_drawish_plot['opening_name']], fontsize=8)
ax3.set_xlabel('Draw Rate (%)', fontsize=11, fontweight='bold')
ax3.set_title('Top 10 Most Drawish Openings', fontsize=12, fontweight='bold')
for i, (idx, row) in enumerate(most_drawish_plot.iterrows()):
    ax3.text(row['draw_rate'], i, f" {row['draw_rate']:.1f}%", va='center', fontsize=9)
sns.despine(ax=ax3)

plt.tight_layout()
plt.savefig('../images/03_opening_effectiveness/top_openings.png', dpi=300, bbox_inches='tight')
plt.show()

## 5. Popularity vs Performance Analysis
Identifying overperforming and underperforming openings relative to their popularity.

In [ ]:
# Filter to openings with at least 30 games for reliable categorization
min_games = 30
categorized_stats = opening_stats[opening_stats['total_games'] >= min_games].copy()

# Calculate White's advantage (white_win_rate - black_win_rate)
categorized_stats['white_advantage'] = categorized_stats['white_win_rate'] - categorized_stats['black_win_rate']

# Identify hidden gems (high white advantage, lower popularity)
# and popularity traps (low white advantage, high popularity)
median_games = categorized_stats['total_games'].median()
median_advantage = categorized_stats['white_advantage'].median()

categorized_stats['category'] = 'Average'
categorized_stats.loc[
    (categorized_stats['total_games'] < median_games) & (categorized_stats['white_advantage'] > median_advantage),
    'category'
] = 'Hidden Gem'
categorized_stats.loc[
    (categorized_stats['total_games'] >= median_games) & (categorized_stats['white_advantage'] < median_advantage),
    'category'
] = 'Popularity Trap'
categorized_stats.loc[
    (categorized_stats['total_games'] >= median_games) & (categorized_stats['white_advantage'] >= median_advantage),
    'category'
] = 'Popular & Strong'

print("="*70)
print(f"POPULARITY VS PERFORMANCE ANALYSIS (≥{min_games} games)")
print("="*70)
print(f"\nOpenings analyzed: {len(categorized_stats)} (of {len(opening_stats)} total)")
print(f"Median games per opening: {median_games:.0f}")
print(f"Median White advantage: {median_advantage:.2f} percentage points")

print(f"\n{'='*70}")
print("CATEGORY DISTRIBUTION")
print("="*70)
category_counts = categorized_stats['category'].value_counts()
print(category_counts)

# Hidden gems - high performance, low popularity
hidden_gems = categorized_stats[categorized_stats['category'] == 'Hidden Gem'].nlargest(10, 'white_advantage')
print(f"\n{'='*70}")
print("TOP 10 HIDDEN GEMS (High Win Rate, Lower Popularity)")
print("="*70)
print(hidden_gems[['opening_name', 'eco_code', 'total_games', 'white_win_rate', 'black_win_rate', 'white_advantage']].to_string(index=False))

# Popularity traps - high popularity, lower performance
popularity_traps = categorized_stats[categorized_stats['category'] == 'Popularity Trap'].nsmallest(10, 'white_advantage')
print(f"\n{'='*70}")
print("TOP 10 POPULARITY TRAPS (High Popularity, Lower Win Rate)")
print("="*70)
print(popularity_traps[['opening_name', 'eco_code', 'total_games', 'white_win_rate', 'black_win_rate', 'white_advantage']].to_string(index=False))

# Visualize popularity vs performance
fig, ax = plt.subplots(figsize=(14, 10))

# Color map for categories
color_map = {
    'Hidden Gem': '#2ecc71',
    'Popularity Trap': '#e74c3c',
    'Popular & Strong': '#3498db',
    'Average': '#95a5a6'
}

for category in categorized_stats['category'].unique():
    data = categorized_stats[categorized_stats['category'] == category]
    ax.scatter(data['total_games'], data['white_advantage'], 
              c=color_map[category], label=category, alpha=0.6, s=100, edgecolors='black', linewidth=0.5)

# Add median lines
ax.axvline(median_games, color='gray', linestyle='--', linewidth=1.5, alpha=0.7, label='Median Popularity')
ax.axhline(median_advantage, color='gray', linestyle='--', linewidth=1.5, alpha=0.7, label='Median Advantage')

# Annotate some interesting points
# Top 3 hidden gems
for idx, row in hidden_gems.head(3).iterrows():
    name = row['opening_name'][:25] + '...' if len(row['opening_name']) > 25 else row['opening_name']
    ax.annotate(name, 
               xy=(row['total_games'], row['white_advantage']),
               xytext=(10, 10), textcoords='offset points',
               fontsize=8, bbox=dict(boxstyle='round,pad=0.3', facecolor='lightgreen', alpha=0.7),
               arrowprops=dict(arrowstyle='->', connectionstyle='arc3,rad=0', lw=1))

# Top 3 popularity traps
for idx, row in popularity_traps.head(3).iterrows():
    name = row['opening_name'][:25] + '...' if len(row['opening_name']) > 25 else row['opening_name']
    ax.annotate(name, 
               xy=(row['total_games'], row['white_advantage']),
               xytext=(10, -15), textcoords='offset points',
               fontsize=8, bbox=dict(boxstyle='round,pad=0.3', facecolor='lightcoral', alpha=0.7),
               arrowprops=dict(arrowstyle='->', connectionstyle='arc3,rad=0', lw=1))

ax.set_xlabel('Number of Games (Popularity)', fontsize=13, fontweight='bold')
ax.set_ylabel('White Advantage (White Win % - Black Win %)', fontsize=13, fontweight='bold')
ax.set_title('Opening Popularity vs Performance', fontsize=15, fontweight='bold', pad=20)
ax.legend(loc='best', fontsize=10)
ax.grid(True, alpha=0.3, linestyle=':')
sns.despine()

plt.tight_layout()
plt.savefig('../images/03_opening_effectiveness/popularity_vs_performance.png', dpi=300, bbox_inches='tight')
plt.show()